# A2 — Knowledge-Base Demo (fill this)
Show OCR quality on a sample and one working retrieval example.

In [2]:
# IMPLEMENT: run OCR quality + one retrieval, end to end


In [2]:
# ==============================================================================
# Comprehensive A2 Demonstration & Evidence Cell
# Executes end-to-end evaluation: OCR metrics (F1/CER/WER), Index stats, and FAISS retrieval.
# ==============================================================================
from __future__ import annotations

import json
import os
import re
import sys
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

# ------------------------------------------------------------------------------
# 1. Environment & Path Setup
# ------------------------------------------------------------------------------
ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "config.yaml").is_file():
    ROOT = ROOT.parent
assert (ROOT / "configs" / "config.yaml").is_file(), "Launch from repository root or notebooks/ directory."
os.chdir(ROOT)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from doc_agent import config, pipeline
from doc_agent.contracts import Page
from doc_agent.index import store
from doc_agent.vision import layout, ocr

CFG = config.load(ROOT / "configs" / "config.yaml")

# ------------------------------------------------------------------------------
# 2. Metric Computation Functions (F1, CER, WER)
# ------------------------------------------------------------------------------
def normalize_text(text: str) -> str:
    """Normalize unicode and collapse redundant whitespace."""
    text = unicodedata.normalize("NFC", text)
    return re.sub(r"\s+", " ", text).strip()

def levenshtein_distance(seq1: list | str, seq2: list | str) -> int:
    """Standard dynamic programming Levenshtein distance."""
    size_x = len(seq1) + 1
    size_y = len(seq2) + 1
    matrix = np.zeros((size_x, size_y), dtype=int)
    for x in range(size_x):
        matrix[x, 0] = x
    for y in range(size_y):
        matrix[0, y] = y

    for x in range(1, size_x):
        for y in range(1, size_y):
            if seq1[x - 1] == seq2[y - 1]:
                matrix[x, y] = matrix[x - 1, y - 1]
            else:
                matrix[x, y] = min(
                    matrix[x - 1, y] + 1,      # Deletion
                    matrix[x, y - 1] + 1,      # Insertion
                    matrix[x - 1, y - 1] + 1   # Substitution
                )
    return int(matrix[size_x - 1, size_y - 1])

def character_metrics(predicted: str, reference: str) -> tuple[float, float, float]:
    """Calculate character-level precision, recall, and F1 based on multiset overlap."""
    pred_clean = normalize_text(predicted)
    ref_clean = normalize_text(reference)
    pred_counts = Counter(pred_clean)
    ref_counts = Counter(ref_clean)
    
    overlap = sum((pred_counts & ref_counts).values())
    total_pred = max(sum(pred_counts.values()), 1)
    total_ref = max(sum(ref_counts.values()), 1)
    
    precision = overlap / total_pred
    recall = overlap / total_ref
    f1 = (2 * precision * recall) / max(precision + recall, 1e-12)
    return precision, recall, f1

def compute_cer(predicted: str, reference: str) -> float:
    """Compute Character Error Rate (CER)."""
    p = normalize_text(predicted)
    r = normalize_text(reference)
    if not r:
        return 0.0 if not p else 1.0
    return levenshtein_distance(p, r) / len(r)

def compute_wer(predicted: str, reference: str) -> float:
    """Compute Word Error Rate (WER)."""
    p_words = normalize_text(predicted).split()
    r_words = normalize_text(reference).split()
    if not r_words:
        return 0.0 if not p_words else 1.0
    return levenshtein_distance(p_words, r_words) / len(r_words)

# ------------------------------------------------------------------------------
# 3. Held-Out OCR Evaluation
# ------------------------------------------------------------------------------
LABELS_PATH = ROOT / "grading_kit" / "labels.jsonl"
assert LABELS_PATH.is_file(), f"Missing ground truth file at {LABELS_PATH}"
LABELS = [json.loads(line) for line in LABELS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]

OCR_SAMPLE_SIZE = min(3, len(LABELS))
ocr_rows = []

start_ocr_time = time.time()
for label in LABELS[:OCR_SAMPLE_SIZE]:
    page_id = label["page_id"]
    image_path = ROOT / "grading_kit" / "heldout_pages" / f"{page_id}.jpg"
    if not image_path.is_file():
        image_path = ROOT / "grading_kit" / "heldout_pages" / f"{page_id}.png"
        
    page = Page(id=page_id, doc_id="krishipath-heldout", image_path=str(image_path))
    page_cfg = dict(CFG)
    
    regions = layout.detect([page], page_cfg)
    prediction = "\n".join(chunk.text for chunk in ocr.transcribe(regions, page_cfg))
    
    prec, rec, f1 = character_metrics(prediction, label["text"])
    cer = compute_cer(prediction, label["text"])
    wer = compute_wer(prediction, label["text"])
    
    ocr_rows.append({
        "page_id": page_id,
        "regions": len(regions),
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "cer": cer,
        "wer": wer,
    })

total_ocr_latency = time.time() - start_ocr_time
mean_prec = float(np.mean([r["precision"] for r in ocr_rows]))
mean_rec = float(np.mean([r["recall"] for r in ocr_rows]))
mean_f1 = float(np.mean([r["f1"] for r in ocr_rows]))
mean_cer = float(np.mean([r["cer"] for r in ocr_rows]))
mean_wer = float(np.mean([r["wer"] for r in ocr_rows]))

display(Markdown(f"""
## Held-out OCR Quality & Error Metrics
- **Evaluated Samples:** `{len(ocr_rows)}` pages (Total latency: `{total_ocr_latency:.2f}s`)
- **Mean Character Precision:** `{mean_prec:.4f}`
- **Mean Character Recall:** `{mean_rec:.4f}`
- **Mean Character F1:** **`{mean_f1:.4f}`**
- **Mean CER (Character Error Rate):** `{mean_cer:.4f}` ({mean_cer * 100:.2f}%)
- **Mean WER (Word Error Rate):** `{mean_wer:.4f}` ({mean_wer * 100:.2f}%)
"""))

print(f"{'Page ID':<12} | {'Regions':<8} | {'Precision':<10} | {'Recall':<10} | {'Char F1':<10} | {'CER':<10} | {'WER':<10}")
print("-" * 80)
for row in ocr_rows:
    print(f"{row['page_id']:<12} | {row['regions']:<8} | {row['precision']:<10.4f} | {row['recall']:<10.4f} | {row['f1']:<10.4f} | {row['cer']:<10.4f} | {row['wer']:<10.4f}")

# ------------------------------------------------------------------------------
# 4. Knowledge Base Inspection & Index Statistics
# ------------------------------------------------------------------------------
index_dir = ROOT / CFG.get("index", {}).get("path", "data/index")
if not (index_dir / "index.faiss").is_file():
    print("Index missing on disk; building knowledge base...")
    pipeline.build_knowledge_base(CFG)

faiss_index, indexed_chunks = store.load(CFG)
metadata = json.loads((index_dir / "metadata.json").read_text(encoding="utf-8"))
indexed_pages = {page_id for chunk in indexed_chunks for page_id in chunk.page_ids}
word_counts = [len(chunk.text.split()) for chunk in indexed_chunks]

display(Markdown(f"""
## Knowledge Base & Vector Index Statistics
- **Index Type:** `{metadata.get('index_type', 'faiss:hnsw')}` (`{metadata.get('metric', 'inner_product')}`)
- **Embedding Model:** `{CFG.get('embed', {}).get('model', 'BAAI/bge-m3')}` (Dimension: `{metadata.get('dimension', faiss_index.d)}`)
- **Total Chunks Indexed:** **`{len(indexed_chunks):,}`**
- **Source Pages Represented:** **`{len(indexed_pages):,} / 378`**
- **HNSW Parameters:** `M={metadata.get('hnsw_m', 32)}`, `efConstruction={metadata.get('ef_construction', 40)}`, `efSearch={metadata.get('ef_search', 64)}`
- **Mean Chunk Length:** `{np.mean(word_counts):.1f} words` (Min: `{np.min(word_counts)}`, Max: `{np.max(word_counts)}`)
"""))

# ------------------------------------------------------------------------------
# 5. Semantic Retrieval Demonstration
# ------------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

QUERY = "কোখ-এর জীবাণু নীতিসমূহ কতটি?"
EXPECTED_PAGE = "page-0036"
TOP_K = 3

embed_cfg = CFG.get("embed", {})
device = "cuda" if str(embed_cfg.get("device", "cpu")).startswith("cuda") else "cpu"
embedder = SentenceTransformer(embed_cfg.get("model", "BAAI/bge-m3"), device=device)

start_search = time.time()
query_vector = embedder.encode([QUERY], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
scores, positions = faiss_index.search(query_vector, k=TOP_K)
search_latency_ms = (time.time() - start_search) * 1000

results = [
    (indexed_chunks[int(position)], float(score))
    for score, position in zip(scores[0], positions[0])
    if position >= 0
]

top_1_pages = set(results[0][0].page_ids) if results else set()
hit_at_1 = EXPECTED_PAGE in top_1_pages

reciprocal_rank = 0.0
for rank, (result, _) in enumerate(results, start=1):
    if EXPECTED_PAGE in result.page_ids:
        reciprocal_rank = 1.0 / rank
        break

display(Markdown(f"""
## Retrieval Demonstration & Ground-Truth Verification
- **Query:** `{QUERY}`
- **Target Ground Truth Page:** `{EXPECTED_PAGE}`
- **Search Latency:** `{search_latency_ms:.2f} ms`
- **Hit@1:** **`{'PASS (True)' if hit_at_1 else 'FAIL (False)'}`**
- **Reciprocal Rank (RR):** `{reciprocal_rank:.4f}`
"""))

for rank, (result, score) in enumerate(results, start=1):
    excerpt = re.sub(r"\s+", " ", result.text).strip()
    preview = excerpt[:250] + ("..." if len(excerpt) > 250 else "")
    print(f"#{rank:<2} | Score={score:.4f} | Pages={result.page_ids} | Chunk={result.id}")
    print(f"    Excerpt: {preview}")
    print("-" * 88)

print(f"Retrieval success at rank 1: {hit_at_1}")


## Held-out OCR Quality & Error Metrics
- **Evaluated Samples:** `3` pages (Total latency: `17.95s`)
- **Mean Character Precision:** `0.9472`
- **Mean Character Recall:** `0.9521`
- **Mean Character F1:** **`0.9496`**
- **Mean CER (Character Error Rate):** `0.2580` (25.80%)
- **Mean WER (Word Error Rate):** `0.4949` (49.49%)


Page ID      | Regions  | Precision  | Recall     | Char F1    | CER        | WER       
--------------------------------------------------------------------------------
page-0035    | 7        | 0.9556     | 0.9448     | 0.9502     | 0.2663     | 0.5139    
page-0036    | 18       | 0.9358     | 0.9498     | 0.9428     | 0.2266     | 0.4837    
page-0037    | 10       | 0.9503     | 0.9615     | 0.9559     | 0.2811     | 0.4870    



## Knowledge Base & Vector Index Statistics
- **Index Type:** `faiss:hnsw` (`inner_product`)
- **Embedding Model:** `BAAI/bge-m3` (Dimension: `1024`)
- **Total Chunks Indexed:** **`1,353`**
- **Source Pages Represented:** **`378 / 378`**
- **HNSW Parameters:** `M=32`, `efConstruction=40`, `efSearch=64`
- **Mean Chunk Length:** `92.3 words` (Min: `11`, Max: `139`)



## Retrieval Demonstration & Ground-Truth Verification
- **Query:** `কোখ-এর জীবাণু নীতিসমূহ কতটি?`
- **Target Ground Truth Page:** `page-0036`
- **Search Latency:** `1378.53 ms`
- **Hit@1:** **`FAIL (False)`**
- **Reciprocal Rank (RR):** `0.0000`


#1  | Score=0.5259 | Pages=['page-0016'] | Chunk=page-0016_c005
    Excerpt: ঁহার এই গবেষণার ফলাফল জীবাণুজনিত রোগ উদ্ভব মতবাদকে আরও সুদৃঢ় করে / নীতিগুলো অনুসরণ করিয়া রোগের প্রকৃত কারণ নির্ধারণ করা হয় তাহা এর যে Koch-এর স্বীকার্য (Koch's postulates) নামে বিশেষভাবে পরচিত Koch এর স্বীকার্য নিম্নেবর্ণিত হলো
----------------------------------------------------------------------------------------
#2  | Score=0.5169 | Pages=['page-0035'] | Chunk=page-0035_c004
    Excerpt: ী জীবাণুজনিত রোগ অনুসন্ধান সম্পর্কে কতকগুলি নীতি ঘোষণা সহকমী তারপর, তাহার ছাত্র এবং পাস্তুরের Robert koch (১৮৭৬ কেন Henle এর
----------------------------------------------------------------------------------------
#3  | Score=0.4621 | Pages=['page-0016'] | Chunk=page-0016_c004
    Excerpt: নি মতবাদ (biogenesis) জীবজনি মতবাদ স্বীকৃতি পাওয়ার আগেই অনেক উদ্ভিদ রোগবিজ্ঞানী কিত্ত ছত্রাককে রোগের কারণ বলিয়া জানিতেন বিরুদ্ধবাদীরা বলিতেন, ছত্রাক আসলে রোগের কারণ নয়, রোগের ফল কিত্তু জীবজনি মতবাদ প্রতিষ্ঠিত হওয়ার পর পরজ্জীবী ছত্র